## **Connexion et chargement de bronze**

In [2]:
# %% Paramètres, connexion et chargement des données de la couche visée
import sys, os, warnings
# sys.path.append("../pipelines")
sys.path.append("/home/jovyan/work/pipelines")
from aq import *
import great_expectations as gx
warnings.filterwarnings("ignore")

COUCHE = os.environ.get("COUCHE", "BRONZE")        # BRONZE aujourd'hui ; SILVER en S4 (US-23, variable COUCHE=SILVER) : la même suite se relance sans modification, seules les requêtes de chargement changent
DOSSIER = "/home/jovyan/work/reports" if os.path.isdir("/home/jovyan/work/reports") else "reports"
os.makedirs(f"{DOSSIER}/ge", exist_ok=True)

# Les colonnes sont ramenées aux mêmes noms dans les deux couches (client_id, telephone, pays...) : une seule suite pour bronze et silver.
# `cle` est la clé primaire de la table (RQ-020) ; `source_systeme` permet de ne viser que les clients du CRM.
CHARGEMENT = {
 "BRONZE": {
  "crm_client":   ("client",   "SELECT client_id AS cle, client_id, 'crm' AS source_systeme, type_personne, telephone, cp, pays FROM bronze.crm_client WHERE _batch_id = :b"),
  "crm_contrat":  ("contrat",  "SELECT contrat_id AS cle, contrat_id, souscripteur_id, date_debut, date_fin, duree_garantie_mois FROM bronze.crm_contrat WHERE _batch_id = :b"),
  "crm_sinistre": ("sinistre", "SELECT sinistre_id AS cle, sinistre_id FROM bronze.crm_sinistre WHERE _batch_id = :b"),
  "cif_clients":  ("cif",      "SELECT column0 AS cle FROM bronze.cif_clients WHERE _batch_id = :b")},
 "SILVER": {
  "client":   ("client",   "SELECT CONCAT(source_systeme, '|', source_id) AS cle, source_id AS client_id, source_systeme, type_personne, telephone_normalise AS telephone, cp, pays_iso AS pays FROM silver.client"),
  "contrat":  ("contrat",  "SELECT contrat_id AS cle, contrat_id, souscripteur_id, date_debut, date_fin, duree_garantie_mois FROM silver.contrat WHERE :b = :b"),
  "sinistre": ("sinistre", "SELECT sinistre_id AS cle, sinistre_id FROM silver.sinistre WHERE :b = :b")}}
NAISSANCES = {"BRONZE": "SELECT client_id, date_naissance FROM bronze.crm_client WHERE _batch_id = :b",
              "SILVER": "SELECT source_id AS client_id, date_naissance FROM silver.client WHERE source_systeme = 'crm' AND :b = :b"}

def age_a(date, naissance):
    """Âge en années révolues à une date (même calcul que TIMESTAMPDIFF(YEAR, naissance, date) en SQL)."""
    a = date.dt.year - naissance.dt.year
    avant = (date.dt.month < naissance.dt.month) | ((date.dt.month == naissance.dt.month) & (date.dt.day < naissance.dt.day))
    return a - avant.astype(int)

def charger(table):
    nature, requete = CHARGEMENT[COUCHE][table]
    df = lire(requete, b=BATCH)
    if nature == "contrat":
        for c in ["date_debut", "date_fin"]:
            df[c] = pd.to_datetime(df[c], errors="coerce")                      # les dates arrivent en texte depuis bronze
        df["duree_garantie_mois"] = pd.to_numeric(df.duree_garantie_mois, errors="coerce")
        nais = lire(NAISSANCES[COUCHE], b=BATCH).drop_duplicates("client_id")
        nais["date_naissance"] = pd.to_datetime(nais.date_naissance, errors="coerce")
        df = df.merge(nais, how="left", left_on="souscripteur_id", right_on="client_id", suffixes=("", "_s"))
        df["age_debut"] = age_a(df.date_debut, df.date_naissance.fillna(df.date_debut))      # RQ-002 : âge à la date de début (US-13)
        df.loc[df.date_naissance.isna(), "age_debut"] = float("nan")
    return nature, df

DONNEES = {table: charger(table) for table in CHARGEMENT[COUCHE]}
for t, (nature, d) in DONNEES.items():
    print(f"{COUCHE.lower()}.{t:<13} {len(d):>7} lignes")
REGLES = lire("SELECT rule_id, severite, seuil_min FROM dq.regle").set_index("rule_id")      # la sévérité et le seuil viennent de dq.regle, pas du code
CODES_PAYS = lire("SELECT code_pays FROM silver.ref_pays").code_pays.tolist()
def mostly(rule_id):
    return float(REGLES.loc[rule_id, "seuil_min"]) / 100


bronze.crm_client     100000 lignes
bronze.crm_contrat    100000 lignes
bronze.crm_sinistre   100000 lignes
bronze.cif_clients     40000 lignes


## **Préparer Great Expectations**

In [3]:
context = gx.get_context(mode="file", project_root_dir=f"{DOSSIER}/ge")
source = context.sources.add_or_update_pandas(name="mariadb_df")

def asset(nom):
    try:
        return source.get_asset(nom)
    except LookupError:
        return source.add_dataframe_asset(name=nom)

def valider(table, nature, df, ajouter):
    br = asset(table).build_batch_request(dataframe=df)
    suite = f"assurconnect_{COUCHE.lower()}_{table}_suite"
    context.add_or_update_expectation_suite(suite)
    v = context.get_validator(batch_request=br, expectation_suite_name=suite)
    ajouter(v)
    v.save_expectation_suite(discard_failed_expectations=False)
    cp = context.add_or_update_checkpoint(name=f"dq_{COUCHE.lower()}_{table}", validations=[{"batch_request": br, "expectation_suite_name": suite}])
    return list(cp.run().run_results.values())[0]["validation_result"].results
print("Great Expectations prêt :", gx.__version__)

Great Expectations prêt : 0.18.19


## **US-10, les règles en tests et les pièges**

In [4]:
SOUSCRIPTEUR_CRM = dict(row_condition='source_systeme == "crm" and type_personne == "souscripteur"', condition_parser="pandas")
FRANCE_CRM       = dict(row_condition='source_systeme == "crm" and pays == "FR"', condition_parser="pandas")
CRM              = dict(row_condition='source_systeme == "crm"', condition_parser="pandas")

def regles_client(v):
    v.expect_column_values_to_not_be_null(column="telephone", mostly=mostly("RQ-003"), meta={"rule_id": "RQ-003"}, **SOUSCRIPTEUR_CRM)
    v.expect_column_values_to_match_regex(column="telephone", regex=r"\S", mostly=mostly("RQ-003"), meta={"rule_id": "RQ-003"}, **SOUSCRIPTEUR_CRM)
    v.expect_column_values_to_match_regex(column="cp", regex=r"^\d{5}$", mostly=mostly("RQ-005"), meta={"rule_id": "RQ-005", "nuls_en_echec": True}, **FRANCE_CRM)
    v.expect_column_values_to_be_in_set(column="pays", value_set=CODES_PAYS, meta={"rule_id": "RQ-007", "nuls_en_echec": True}, **CRM)
    v.expect_column_values_to_be_in_set(column="type_personne", value_set=["prospect", "souscripteur", "assure"], meta={"rule_id": "RQ-009", "nuls_en_echec": True}, **CRM)

def regles_contrat(v):
    v.expect_column_pair_values_A_to_be_greater_than_B(column_A="date_fin", column_B="date_debut", or_equal=False, meta={"rule_id": "RQ-001"})
    v.expect_column_values_to_be_between(column="age_debut", min_value=18, meta={"rule_id": "RQ-002"})
    v.expect_column_values_to_be_in_set(column="duree_garantie_mois", value_set=[12, 24, 36, 60], meta={"rule_id": "RQ-010", "nuls_en_echec": True})

def cle_primaire(v):
    v.expect_column_values_to_not_be_null(column="cle", meta={"rule_id": "RQ-020"})
    v.expect_column_values_to_be_unique(column="cle", meta={"rule_id": "RQ-020"})

SUITES = {"client": [regles_client, cle_primaire], "contrat": [regles_contrat, cle_primaire], "sinistre": [cle_primaire], "cif": [cle_primaire]}
def tout_ajouter(nature):
    return lambda v: [f(v) for f in SUITES[nature]]

mesures_ge = []
for table, (nature, df) in DONNEES.items():
    resultats = valider(table, nature, df, tout_ajouter(nature))
    par_regle = {}
    for r in resultats:
        m = r.expectation_config.meta
        n, ko = r.result["element_count"], r.result["unexpected_count"] + (r.result.get("missing_count", 0) if m.get("nuls_en_echec") else 0)
        t0, k0 = par_regle.get(m["rule_id"], (n, 0))
        par_regle[m["rule_id"]] = (t0, k0 + ko)
    for rule, (n, ko) in par_regle.items():
        mesures_ge.append({"rule_id": rule, "table": table, "testees": int(n), "echecs": int(ko)})
bilan = pd.DataFrame(mesures_ge).sort_values(["rule_id", "table"])
bilan["conformite_pct"] = (100 * (bilan.testees - bilan.echecs) / bilan.testees).round(3)
bilan["seuil_pct"] = [float(REGLES.loc[r, "seuil_min"]) for r in bilan.rule_id]
bilan["severite"] = [REGLES.loc[r, "severite"] for r in bilan.rule_id]
bilan["verdict"] = ["conforme" if c >= s else "sous le seuil" for c, s in zip(bilan.conformite_pct, bilan.seuil_pct)]
print(bilan.to_string(index=False))
context.build_data_docs()

# ===== À vous : les pièges, chiffre avant / après correction
_, cl = DONNEES["crm_client"]
sousc = cl[cl.type_personne == "souscripteur"]
print("\nRQ-003 (souscripteurs :", len(sousc), ") | test IS NULL seul :", int(sousc.telephone.isna().sum()),
      "| nul + vide + blanc :", int((sousc.telephone.isna() | (sousc.telephone.fillna("").str.strip() == "")).sum()))
fr = cl[cl.pays == "FR"]
print("RQ-005 | contrôle sur toutes les adresses :", int((~cl.cp.fillna("").str.fullmatch(r"\d{5}")).sum()), "sur", len(cl),
      "| sur les adresses FR seulement :", int((~fr.cp.fillna("").str.fullmatch(r"\d{5}")).sum()), "sur", len(fr))
print("RQ-009 | comparaison exacte, hors domaine :", int((~cl.type_personne.isin(["prospect","souscripteur","assure"])).sum()),
      "| après minuscules et sans accent :", int((~cl.type_personne.str.lower().str.replace("é","e").isin(["prospect","souscripteur","assure"])).sum()))

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/13 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/13 [00:00<?, ?it/s]

rule_id        table  testees  echecs  conformite_pct  seuil_pct      severite       verdict
 RQ-001  crm_contrat   100000     800          99.200      100.0     BLOQUANTE sous le seuil
 RQ-002  crm_contrat   100000     301          99.699      100.0     BLOQUANTE sous le seuil
 RQ-003   crm_client    49898    5989          87.998       98.0 AVERTISSEMENT sous le seuil
 RQ-005   crm_client    95437    1456          98.474       98.0 AVERTISSEMENT      conforme
 RQ-007   crm_client   100000    3050          96.950      100.0     BLOQUANTE sous le seuil
 RQ-009   crm_client   100000     200          99.800      100.0     BLOQUANTE sous le seuil
 RQ-010  crm_contrat   100000     600          99.400      100.0     BLOQUANTE sous le seuil
 RQ-020  cif_clients    40000       0         100.000      100.0     BLOQUANTE      conforme
 RQ-020   crm_client   100000       0         100.000      100.0     BLOQUANTE      conforme
 RQ-020  crm_contrat   100000       0         100.000      100.0     B

## **US-11, lire les lignes en échec**

In [5]:
COMPLET = lambda ident: {"result_format": "COMPLETE", "unexpected_index_column_names": [ident]}

def lignes_en_echec(table, ident, expectation, **kwargs):
    nature, df = DONNEES[table]
    context.add_or_update_expectation_suite("assurconnect_echecs")
    v = context.get_validator(batch_request=asset(f"echecs_{table}").build_batch_request(dataframe=df), expectation_suite_name="assurconnect_echecs")
    return getattr(v, expectation)(result_format=COMPLET(ident), **kwargs).result.get("unexpected_index_list", [])

echecs = []
t = "crm_client"
for rule, colonne, motif_txt, exp, kw in [
    ("RQ-003", "telephone", "téléphone nul, vide ou blanc", "expect_column_values_to_not_be_null", SOUSCRIPTEUR_CRM),
    ("RQ-003", "telephone", "téléphone nul, vide ou blanc", "expect_column_values_to_match_regex", dict(regex=r"\S", **SOUSCRIPTEUR_CRM)),
    ("RQ-005", "cp", "code postal : pas 5 chiffres", "expect_column_values_to_match_regex", dict(regex=r"^\d{5}$", **FRANCE_CRM)),
    ("RQ-009", "type_personne", "type_personne hors domaine", "expect_column_values_to_be_in_set", dict(value_set=["prospect", "souscripteur", "assure"], **CRM))]:
    for ligne in lignes_en_echec(t, "client_id", exp, column=colonne, **kw):
        echecs.append({"rule_id": rule, "table": t, "id": ligne["client_id"], "valeur": ligne.get(colonne), "motif": motif_txt})
t = "crm_contrat"
for ligne in lignes_en_echec(t, "contrat_id", "expect_column_values_to_be_in_set", column="duree_garantie_mois", value_set=[12, 24, 36, 60]):
    echecs.append({"rule_id": "RQ-010", "table": t, "id": ligne["contrat_id"], "valeur": ligne.get("duree_garantie_mois"), "motif": "durée de garantie hors liste"})
for ligne in lignes_en_echec("crm_contrat", "contrat_id", "expect_column_pair_values_A_to_be_greater_than_B", column_A="date_fin", column_B="date_debut", or_equal=False):
    echecs.append({"rule_id": "RQ-001", "table": "crm_contrat", "id": ligne["contrat_id"], "valeur": f"{ligne.get('date_fin')} <= {ligne.get('date_debut')}", "motif": "date de fin non postérieure au début"})

echecs = pd.DataFrame(echecs)
os.makedirs(f"{DOSSIER}/echecs", exist_ok=True)
for rule, sous in echecs.groupby("rule_id"):
    sous.to_csv(f"{DOSSIER}/echecs/lignes_en_echec_{rule}.csv", index=False)
print(echecs.groupby(["rule_id", "table"]).size().rename("lignes_en_echec"))
print("\nMotifs distincts de RQ-009 :\n", echecs[echecs.rule_id == "RQ-009"].valeur.value_counts())

# ===== À vous : échantillon de 20 lignes, toutes règles confondues, en tableau prêt à coller dans le livrable
ech = echecs.groupby("rule_id", group_keys=False).apply(lambda g: g.sample(min(len(g), 4), random_state=1)).head(20)
print("\n| Règle | Table | Identifiant | Valeur | Motif |\n|---|---|---|---|---|")
for _, r in ech.iterrows():
    print(f"| {r.rule_id} | {r.table} | {r.id} | {r.valeur} | {r.motif} |")

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/9 [00:00<?, ?it/s]

rule_id  table      
RQ-001   crm_contrat     800
RQ-003   crm_client     5989
RQ-005   crm_client     1456
RQ-009   crm_client      200
RQ-010   crm_contrat     600
Name: lignes_en_echec, dtype: int64

Motifs distincts de RQ-009 :
 valeur
client          60
Souscripteur    50
assuré          40
SOUSCRIPTEUR    30
prosp.          20
Name: count, dtype: int64

| Règle | Table | Identifiant | Valeur | Motif |
|---|---|---|---|---|
| RQ-001 | crm_contrat | CT0000349 | 2014-08-12 00:00:00 <= 2014-08-12 00:00:00 | date de fin non postérieure au début |
| RQ-001 | crm_contrat | CT0020055 | 2021-12-18 00:00:00 <= 2021-12-20 00:00:00 | date de fin non postérieure au début |
| RQ-001 | crm_contrat | CT0006937 | 2018-09-16 00:00:00 <= 2018-09-16 00:00:00 | date de fin non postérieure au début |
| RQ-001 | crm_contrat | CT0037757 | 2024-09-16 00:00:00 <= 2024-09-16 00:00:00 | date de fin non postérieure au début |
| RQ-003 | crm_client | S-4733 |  | téléphone nul, vide ou blanc |
| RQ-003 | crm_c

## **US-13, l’âge à la date de début du contrat (RQ-002)**

In [6]:
nature, ct = DONNEES["crm_contrat"]
c = ct[ct.age_debut.notna()].copy()
ref = pd.Timestamp(DATE_REF)
c["age_reference"] = age_a(pd.Series(ref, index=c.index), c.date_naissance)
mineur_signature, mineur_reference = c.age_debut < 18, c.age_reference < 18
print(f"contrats avec souscripteur connu : {len(c)} | mineurs à la signature : {int(mineur_signature.sum())} | "
      f"mineurs à la date de référence : {int(mineur_reference.sum())} | verdict qui change : {int((mineur_signature != mineur_reference).sum())}")
print(c[mineur_signature != mineur_reference][["contrat_id", "souscripteur_id", "date_naissance", "date_debut", "age_debut", "age_reference"]].head(10).to_string(index=False))

contrats avec souscripteur connu : 92996 | mineurs à la signature : 301 | mineurs à la date de référence : 157 | verdict qui change : 144
contrat_id souscripteur_id date_naissance date_debut  age_debut  age_reference
 CT0000387         A-22244     1996-09-28 2014-09-01       17.0             30
 CT0001356          A-2454     1998-10-17 2016-02-28       17.0             27
 CT0002683         S-21318     1999-07-06 2017-03-05       17.0             27
 CT0002784         A-30785     2000-07-18 2017-03-27       16.0             26
 CT0003211         H-15141     2001-03-23 2017-06-22       16.0             25
 CT0004249         S-14454     2001-12-29 2018-01-06       16.0             24
 CT0005613         H-30584     2001-02-16 2018-05-14       17.0             25
 CT0006254          A-8673     2000-12-29 2018-07-13       17.0             25
 CT0009140         A-13520     2002-12-12 2019-04-03       16.0             23
 CT0010269         A-24682     2002-12-17 2019-07-19       16.0         

## **US-15, la mesure de référence enregistrée**

In [7]:
for m in mesures_ge:
    mesurer(COUCHE, m["rule_id"], m["table"], m["testees"], m["echecs"])
lecture = mesures(COUCHE)
lecture["verdict"] = ["conforme" if t >= s else "sous le seuil" for t, s in zip(lecture.taux_conformite.astype(float), lecture.seuil_min.astype(float))]
print(lecture.to_string(index=False))
print("\nSynthèse par dimension :")
print(lire("SELECT dimension, taux_moyen FROM dq.dq_kpi_dimension WHERE batch_id = :b AND couche = :c ORDER BY taux_moyen", b=BATCH, c=COUCHE).to_string(index=False))

couche rule_id      severite      table_cible  nb_testees  nb_echecs taux_conformite seuil_min       verdict
BRONZE  RQ-001     BLOQUANTE      crm_contrat      100000        800          99.200    100.00 sous le seuil
BRONZE  RQ-002     BLOQUANTE      crm_contrat      100000        301          99.699    100.00 sous le seuil
BRONZE  RQ-003 AVERTISSEMENT       crm_client       49898       5989          87.998     98.00 sous le seuil
BRONZE  RQ-005 AVERTISSEMENT       crm_client       95437       1456          98.474     98.00      conforme
BRONZE  RQ-007     BLOQUANTE       crm_client      100000       3050          96.950    100.00 sous le seuil
BRONZE  RQ-009     BLOQUANTE       crm_client      100000        200          99.800    100.00 sous le seuil
BRONZE  RQ-010     BLOQUANTE      crm_contrat      100000        600          99.400    100.00 sous le seuil
BRONZE  RQ-020     BLOQUANTE      cif_clients       40000          0         100.000    100.00      conforme
BRONZE  RQ-020     